# Qwen3-8B Answer Evaluation

## Purpose

This notebook evaluates the answers generated by `08_qwen_rag.ipynb` on the **test** split, for each of the three retrievers (TF-IDF, BM25, dense).

## Metrics

Since answers are free-form (not extractive spans), three complementary metrics are used:

- **Exact Match (EM):** 1 if the normalized generated answer equals the normalized reference answer, else 0. Expected to be low/near-zero for free-form answers; kept for completeness.
- **Token F1:** harmonic mean of precision and recall of overlapping tokens between the generated and reference answer (SQuAD-style).
- **ROUGE-L F1:** based on the longest common subsequence between the generated and reference answer, capturing overlap in order as well as content.

All three are computed per question and then averaged per retriever.

In [1]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
import altair as alt
from sentence_transformers import SentenceTransformer

In [2]:
GENERATOR_NAME = "qwen"

RETRIEVERS = ("tfidf", "bm25", "dense")


#SPLIT = "validation"
SPLIT = "test"

# Samo za SPLIT="test": "original" ili "concise".
# Ignoriše se za SPLIT="validation".
PROMPT_VARIANT = "original"

PROJECT_ROOT = Path.cwd()

GENERATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "generation"
    / "qwen_colab"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "qwen"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert GENERATION_DIR.exists(), f"Ne postoji: {GENERATION_DIR}"

print("Generisani odgovori:", GENERATION_DIR)
print("Rezultati evaluacije:", RESULTS_DIR)
print("Split:", SPLIT, "| Prompt varijanta:", PROMPT_VARIANT if SPLIT == "test" else "original (jedina za validaciju)")

Generisani odgovori: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/generation/qwen_colab
Rezultati evaluacije: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/evaluation/qwen
Split: test | Prompt varijanta: concise


In [3]:
def load_jsonl(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Neispravan JSON u redu {line_number}: {path}"
                ) from error

    return records

## Loading Generated Answers


In [4]:
required_fields = {
    "question_id",
    "question",
    "answer",
    "retriever",
    "generator",
    "split",
    "generated_answer",
}

generation_data = {}

for retriever in RETRIEVERS:
    if SPLIT == "validation":
        filename = f"{retriever}_{SPLIT}_answers.jsonl"
    else:
        filename = f"{retriever}_{SPLIT}_{PROMPT_VARIANT}_answers.jsonl"

    path = GENERATION_DIR / filename

    if not path.exists():
        print(f"Preskočeno: {path} ne postoji.")
        continue

    records = load_jsonl(path)

    for record in records:
        missing = required_fields - record.keys()
        if missing:
            raise ValueError(
                f"Pitanje {record.get('question_id')} u {path} nema polja: {sorted(missing)}"
            )

    generation_data[retriever] = records
    print(f"{retriever}/{SPLIT}/{PROMPT_VARIANT if SPLIT=='test' else '-'}: {len(records)} odgovora")

if not generation_data:
    raise FileNotFoundError(
        f"Nisu pronađeni generisani odgovori za split '{SPLIT}' ni za jedan retriever."
    )

tfidf/test/concise: 22 odgovora
bm25/test/concise: 22 odgovora
dense/test/concise: 22 odgovora


## Text Normalization and Metrics


In [5]:
TOKEN_PATTERN = re.compile(r"[\w]+", re.UNICODE)


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str):
    return TOKEN_PATTERN.findall(normalize_text(text))

In [6]:
def compute_exact_match(prediction: str, reference: str) -> int:
    return int(normalize_text(prediction) == normalize_text(reference))


def compute_token_f1(prediction: str, reference: str) -> float:
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)

    if not pred_tokens and not ref_tokens:
        return 1.0

    if not pred_tokens or not ref_tokens:
        return 0.0

    from collections import Counter

    pred_counts = Counter(pred_tokens)
    ref_counts = Counter(ref_tokens)

    overlap = sum(
        min(pred_counts[token], ref_counts[token])
        for token in pred_counts
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)

In [7]:
def longest_common_subsequence_length(a: list, b: list) -> int:
    previous_row = [0] * (len(b) + 1)

    for token_a in a:
        current_row = [0] * (len(b) + 1)

        for j, token_b in enumerate(b, start=1):
            if token_a == token_b:
                current_row[j] = previous_row[j - 1] + 1
            else:
                current_row[j] = max(previous_row[j], current_row[j - 1])

        previous_row = current_row

    return previous_row[-1]


def compute_rouge_l(prediction: str, reference: str) -> float:
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)

    if not pred_tokens and not ref_tokens:
        return 1.0

    if not pred_tokens or not ref_tokens:
        return 0.0

    lcs_length = longest_common_subsequence_length(pred_tokens, ref_tokens)

    if lcs_length == 0:
        return 0.0

    precision = lcs_length / len(pred_tokens)
    recall = lcs_length / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)

SEMANTIC_MODEL_NAME = "intfloat/multilingual-e5-base"

semantic_model = SentenceTransformer(SEMANTIC_MODEL_NAME)


def compute_semantic_similarity(predictions: list, references: list) -> list:
    prefixed_predictions = ["query: " + text.strip() for text in predictions]
    prefixed_references = ["query: " + text.strip() for text in references]

    prediction_embeddings = semantic_model.encode(
        prefixed_predictions,
        batch_size=16,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    reference_embeddings = semantic_model.encode(
        prefixed_references,
        batch_size=16,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    similarities = (prediction_embeddings * reference_embeddings).sum(axis=1)

    return similarities.tolist()

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

## Per-Question and Aggregate Metrics

In [8]:
def evaluate_generations(records: list, retriever: str) -> pd.DataFrame:
    predictions = [record["generated_answer"] for record in records]
    references = [record["answer"] for record in records]

    semantic_similarities = compute_semantic_similarity(predictions, references)

    rows = []

    for record, prediction, reference, semantic_sim in zip(
        records, predictions, references, semantic_similarities
    ):
        rows.append({
            "question_id": record["question_id"],
            "retriever": retriever,
            "split": record["split"],
            "EM": compute_exact_match(prediction, reference),
            "F1": compute_token_f1(prediction, reference),
            "ROUGE_L": compute_rouge_l(prediction, reference),
            "Semantic_sim": semantic_sim,
        })

    return pd.DataFrame(rows)


per_question_frames = [
    evaluate_generations(records, retriever)
    for retriever, records in generation_data.items()
]

per_question_df = pd.concat(per_question_frames, ignore_index=True)

per_question_df.head()

,question_id,retriever,split,EM,F1,ROUGE_L,Semantic_sim
0,61,tfidf,test,0,0.367347,0.244898,0.941083
1,25,tfidf,test,0,0.231579,0.168421,0.864041
2,27,tfidf,test,0,0.458333,0.416667,0.962789
3,130,tfidf,test,0,0.149254,0.149254,0.892010
4,33,tfidf,test,0,0.200000,0.133333,0.908135


In [9]:
metrics_df = (
    per_question_df
    .groupby("retriever")[["EM", "F1", "ROUGE_L", "Semantic_sim"]]
    .mean()
    .reset_index()
    .sort_values("F1", ascending=False)
)

metrics_df

,retriever,EM,F1,ROUGE_L,Semantic_sim
1,dense,0.0,0.323050,0.278028,0.904510
2,tfidf,0.0,0.288707,0.233561,0.905888
0,bm25,0.0,0.288046,0.248253,0.903693


## Comparing Retrievers on Validation

In [10]:
metrics_plot_df = metrics_df.melt(
    id_vars="retriever",
    value_vars=["EM", "F1", "ROUGE_L"],
    var_name="metric",
    value_name="score",
)

alt.Chart(metrics_plot_df).mark_bar().encode(
    x=alt.X("retriever:N", title="Retriever"),
    y=alt.Y("score:Q", title="Score", scale=alt.Scale(domain=[0, 1])),
    color=alt.Color("metric:N", title="Metrika"),
    xOffset="metric:N",
    tooltip=["retriever", "metric", alt.Tooltip("score:Q", format=".3f")],
).properties(
    title=f"Qwen3-8B — poređenje retrievera na {SPLIT} skupu",
    width=500,
    height=350,
)

alt.Chart(...)

## Inspecting Individual Answers


In [11]:
best_retriever = metrics_df.iloc[0]["retriever"]

sample_df = (
    per_question_df[per_question_df["retriever"] == best_retriever]
    .sort_values("F1")
    .head(5)
)

sample_records = {
    record["question_id"]: record
    for record in generation_data[best_retriever]
}

for _, row in sample_df.iterrows():
    record = sample_records[row["question_id"]]
    print(f"Pitanje: {record['question']}")
    print(f"Referentni odgovor: {record['answer']}")
    print(f"Generisani odgovor: {record['generated_answer']}")
    print(f"EM={row['EM']:.0f}  F1={row['F1']:.3f}  ROUGE_L={row['ROUGE_L']:.3f}")
    print("-" * 80)

Pitanje: Navesti primer sistemskog testiranja pri testiranju softvera.
Referentni odgovor: Kod elektronskog bankarstva može se testirati ceo tok prenosa novca, uključujući različite ulaze, granične slučajeve i proveru tačnosti, pouzdanosti i bezbednosti sistema.
Generisani odgovor: Testiranje pouzdanosti tokom 72h, gde se sistem testira u trajanju od 72 sata bez prekida, u realnim uslovima korišćenja.
EM=0  F1=0.098  ROUGE_L=0.049
--------------------------------------------------------------------------------
Pitanje: Navesti primer testova pouzdanosti pri testiranju softvera.
Referentni odgovor: Aplikacija za elektronsko bankarstvo može se izvršavati 72 sata uz stalne zahteve i periodične simulacije grešaka, pri čemu se prate padovi, oporavak, memorija, performanse i konzistentnost podataka.
Generisani odgovor: Testovi pouzdanosti uključuju test slučajevi kao što je TC-P006, gde se sistem testira tokom 72 sati bez prekida, uz konstantno generisanje korisničkih zahteva, kako bi se otk

In [12]:
suffix = f"{SPLIT}_{PROMPT_VARIANT}" if SPLIT == "test" else SPLIT

per_question_df.to_csv(
    RESULTS_DIR / f"{GENERATOR_NAME}_{suffix}_per_question_metrics.csv",
    index=False,
)

metrics_df.to_csv(
    RESULTS_DIR / f"{GENERATOR_NAME}_{suffix}_metrics.csv",
    index=False,
)

metadata = {
    "generator": GENERATOR_NAME,
    "split": SPLIT,
    "prompt_variant": PROMPT_VARIANT if SPLIT == "test" else "original",
    "retrievers_evaluated": list(generation_data.keys()),
    "metrics": ["EM", "F1", "ROUGE_L"],
    "n_questions_per_retriever": {
        retriever: len(records)
        for retriever, records in generation_data.items()
    },
}

with (RESULTS_DIR / f"{GENERATOR_NAME}_{suffix}_metadata.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print("Sačuvani rezultati evaluacije u:", RESULTS_DIR)

Sačuvani rezultati evaluacije u: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/evaluation/qwen


In [13]:
qwen_summary_files = {
    "validation": RESULTS_DIR / "qwen_validation_metrics.csv",
    "test_original": RESULTS_DIR / "qwen_test_original_metrics.csv",
    "test_concise": RESULTS_DIR / "qwen_test_concise_metrics.csv",
}

qwen_summary_frames = []

for run_name, path in qwen_summary_files.items():
    if not path.exists():
        print(f"Preskočeno: {path} ne postoji.")
        continue

    df = pd.read_csv(path)
    df.insert(0, "run", run_name)
    qwen_summary_frames.append(df)

qwen_summary_df = pd.concat(qwen_summary_frames, ignore_index=True)

qwen_summary_df

,run,retriever,EM,F1,ROUGE_L,Semantic_sim
0,validation,dense,0.0,0.280126,0.202783,NaN
1,validation,bm25,0.0,0.269089,0.201277,NaN
2,validation,tfidf,0.0,0.252122,0.192527,NaN
3,test_original,dense,0.0,0.285513,0.245146,NaN
4,test_original,bm25,0.0,0.274308,0.218996,NaN
5,test_original,tfidf,0.0,0.260952,0.208733,NaN
6,test_concise,dense,0.0,0.323050,0.278028,0.904510
7,test_concise,tfidf,0.0,0.288707,0.233561,0.905888
8,test_concise,bm25,0.0,0.288046,0.248253,0.903693


In [14]:
alt.Chart(qwen_summary_df).mark_bar().encode(
    x=alt.X("retriever:N", title="Retriever"),
    y=alt.Y("F1:Q", title="F1", scale=alt.Scale(domain=[0, 1])),
    color=alt.Color("run:N", title="Run"),
    xOffset="run:N",
    tooltip=["run", "retriever", alt.Tooltip("F1:Q", format=".3f"), alt.Tooltip("ROUGE_L:Q", format=".3f")],
).properties(
    title="Qwen3-8B — validacija vs test (original/concise prompt)",
    width=550,
    height=350,
)

alt.Chart(...)

## Qwen3-8B — zaključak

- Poredak retrievera je dosledan između validacije i testa: **dense > bm25 ≈ tfidf**, na obe metrike (F1, ROUGE-L).
- **Concise prompt dosledno poboljšava F1/ROUGE-L za sva tri retrievera** u odnosu na original prompt (F1 +0.014 do +0.037)
- Finalna konfiguracija: **dense retriever + concise prompt** (F1=0.323, ROUGE-L=0.278 na testu).
- Semantička sličnost (embedding cosine, isti model kao dense retriever) je visoka i ujednačena za sva tri retrievera (0.90-0.91), za concise prompt na testu** — za razliku od F1/ROUGE-L, ne pravi jasnu razliku između retrievera. Ovo je delom očekivano ograničenje ove metrike (kompresovan opseg vrednosti kod tematski sličnih tekstova), pa se koristi kao dopunska potvrda da odgovori nisu potpuno van teme, ne kao primarni kriterijum 
- Preostalo ograničenje: model ponekad generiše sadržajno uverljive ali generičke odgovore kada tačan kontekst (npr. konkretni imenovani primeri) nije u top-k retrieved chunkovima — ovo je do retrievala ne generatora.